# Linear Regression

## Objectives
The objective of this notebook is to summarise the theory of linear regression before modelling, because it shapes the decisions made along the way. We cover the model and its solution, the assumptions, bias and variance, the Gauss–Markov theorem, hypothesis testing and goodness of fit. Here $n$ is the number of houses, $k = p + 1$ is the number of coefficients ($p$ features and an intercept), and $X$ is the $n \times k$ matrix whose first column is ones.

## 1. The Model
The model is
$$y = X\beta + \varepsilon$$
- $\beta$ are fixed, unknown population coefficients.
- $\varepsilon$ is everything the features do not explain.

### Solution
Ordinary least squares chooses $b$ to minimise the sum of squared residuals $S(b) = (y - Xb)^\top (y - Xb)$. Setting the gradient to zero gives the normal equations:
$$\frac{\partial S}{\partial b} = -2X^\top y + 2X^\top X b = 0 \;\Rightarrow\; X^\top X \hat\beta = X^\top y$$
If the columns of $X$ are linearly independent, $X^\top X$ is invertible and
$$\hat\beta = (X^\top X)^{-1} X^\top y$$
- The second derivative $2X^\top X$ is positive definite, so this is the unique minimum.
- Geometrically, $\hat y = X\hat\beta$ is the projection of $y$ onto the column space of $X$. The residuals $e = y - X\hat\beta$ are orthogonal to every column, so with an intercept they sum to zero.
- Computing $\hat\beta$ needs no assumptions. The assumptions are about what $\hat\beta$ tells us about $\beta$.
- Without full rank, $\hat\beta$ is not unique, but the fitted values are.

## 2. Assumptions
The properties of $\hat\beta$ below rest on these assumptions about $X$ and $\varepsilon$:
- The columns of $X$ are linearly independent (full column rank, $n \ge k$), so $\hat\beta$ is unique.
- The relationship is linear, $E[y \mid X] = X\beta$, which means the model is correctly specified.
- The errors are exogenous, $E[\varepsilon \mid X] = 0$. They are unrelated to the features, and with an intercept any constant mean is absorbed.
- The errors are homoskedastic, $\text{Var}(\varepsilon_i \mid X) = \sigma^2$. The variance is constant and does not depend on the features.
- The errors are uncorrelated, $\text{Cov}(\varepsilon_i, \varepsilon_j \mid X) = 0$ for $i \ne j$. Together with the previous point, $\text{Var}(\varepsilon \mid X) = \sigma^2 I$.
- The errors are normal, $\varepsilon \sim N(0, \sigma^2 I)$. This is the only optional assumption, and it is needed only for exact small-sample inference.

## 3. Bias, Variance and the Gauss–Markov Theorem
Substituting $y = X\beta + \varepsilon$ gives
$$\hat\beta = \beta + (X^\top X)^{-1} X^\top \varepsilon$$
The error in $\hat\beta$ is a weighted sum of the noise. A different sample of houses has different noise, so $\hat\beta$ is random, and its properties follow from how $\varepsilon$ behaves.

### Bias
$$E[\hat\beta \mid X] - \beta = (X^\top X)^{-1} X^\top E[\varepsilon \mid X]$$
- This is zero under exogeneity. If part of the noise moves with a feature, OLS credits that feature with an effect that belongs to the noise.
- Fitting a line to a curve leaves the curve in $\varepsilon$, so misspecification is the same problem. It biases the predictions as well as the coefficients.

### Variance
$$\text{Var}(\hat\beta \mid X) = (X^\top X)^{-1} X^\top \,\text{Var}(\varepsilon \mid X)\, X (X^\top X)^{-1} = \sigma^2 (X^\top X)^{-1} \quad \text{if } \text{Var}(\varepsilon \mid X) = \sigma^2 I$$
- It grows with the noise $\sigma^2$, and shrinks with more rows and more spread in the features.
- It grows when columns are nearly collinear, which is what the variance inflation factor (VIF) measures.

### Gauss–Markov Theorem
- Under linearity, exogeneity, full rank, constant variance and uncorrelated errors, OLS is the best linear unbiased estimator (BLUE).
- For any other linear unbiased estimator $\tilde\beta = Cy$, the matrix $\text{Var}(\tilde\beta) - \text{Var}(\hat\beta)$ is positive semidefinite, so every combination $c^\top \hat\beta$ has the smallest variance.
- Normality is not needed. Any other such estimator is OLS plus a term that is invisible to the features, which adds only variance.

##### Implications:
- Best is only among unbiased estimators. If the variances differ or the errors are correlated, OLS is still unbiased but not best, and its standard errors are wrong. Weighted or generalised least squares weights each house by the inverse of its error variance.
- Ridge and lasso are biased, so they fall outside the theorem and can have lower total error. This is the bias–variance trade-off.
- The theorem says nothing about whether the model is right. Linearity and exogeneity decide the bias.
- The variance of a prediction at a new house $x_0$ is $\sigma^2 + \sigma^2 x_0^\top (X^\top X)^{-1} x_0$, which is larger for houses unlike the training data.

## 4. Hypothesis Testing
Under normal errors, $\hat\beta \sim N(\beta, \sigma^2 (X^\top X)^{-1})$, and $\sigma^2$ is estimated by $\hat\sigma^2 = e^\top e / (n - k)$.

### t-test
This tests one coefficient:
$$t_j = \frac{\hat\beta_j - \beta_{j0}}{\text{se}(\hat\beta_j)}, \qquad \text{se}(\hat\beta_j) = \hat\sigma \sqrt{[(X^\top X)^{-1}]_{jj}} \;\sim\; t_{n-k} \text{ under } H_0$$
- The usual null is $H_0: \beta_j = 0$. It asks whether the feature adds anything given the other features in the model.
- A confidence interval is $\hat\beta_j \pm t_{n-k,\,1-\alpha/2}\, \text{se}(\hat\beta_j)$.

### F-test
This tests $q$ restrictions on the coefficients:
$$F = \frac{(\text{RSS}_r - \text{RSS}_u)/q}{\text{RSS}_u/(n-k)} \;\sim\; F_{q,\,n-k} \text{ under } H_0$$
- $\text{RSS}_r$ is from the restricted model and $\text{RSS}_u$ from the full model. It tests whether a group of coefficients, such as all the dummies of one category, is jointly zero.
- With all $p$ slopes restricted to zero it is the overall test, and with $q = 1$ it equals $t^2$.

##### When the tests can be trusted:
- Without normality they are approximately valid for large $n$, by the central limit theorem.
- Without constant variance or uncorrelated errors the standard errors are wrong, so the tests and intervals are wrong. Robust standard errors are one fix.
- Ridge and lasso give biased coefficients, so these tests no longer apply.

## 5. Goodness of Fit
The coefficient of determination and its adjusted version are
$$R^2 = 1 - \frac{\text{RSS}}{\text{TSS}}, \qquad \bar R^2 = 1 - \frac{\text{RSS}/(n-k)}{\text{TSS}/(n-1)}$$
- $R^2$ is the share of the variance of $y$ that the model explains. It needs an intercept to be defined this way.
- It never falls when a column is added, even a useless one. Adjusted $\bar R^2$ penalises the extra coefficients.
- It is measured on the training data, so it says nothing about prediction on new houses. A high $R^2$ is also compatible with a misspecified model.
- For prediction we use cross-validated error, which measures performance on houses the model has not seen.